# Day 2 — GT-Mean Loss gated fine-tune (Kaggle T4 x2)

One decision only: does **GT-Mean Loss** beat plain L1 when both fine-tune the *same official
LOL-v2-syn checkpoint* on LOL-v2-syn train? Six short matched runs, 3 seeds x 2 losses, 10k iterations.

**Gate (decided in advance):** GT-Mean must beat the L1 control by **>0.1 dB mean best validation PSNR**.
- Gate passes -> Day 3/4: longer fine-tune (40-60k) with checkpoint averaging + self-ensemble.
- Gate fails -> STOP. Keep the Day 1 no-training result (25.9080 dB) as the final paper. No tuning sigma, no test peeking.

Selection uses **validation PSNR only** (10% deterministic split of the training pairs, seed 2026).
The 100-image held-out test set is never touched in this notebook.

**Kaggle settings:** GPU T4 x2, Internet ON, **Save & Run All**. Download `gtmean_gate_state.zip` and `gate.json`
from the Output tab when it finishes; if a session stops, attach the state zip and run again.


In [ ]:
import os, sys, subprocess, glob, json, zipfile
from pathlib import Path

os.chdir('/kaggle/working')
if not Path('btp').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Crazycraftie/low-light-enhancement-btp.git','btp'],check=True)
os.chdir('/kaggle/working/btp')
print('HEAD:', subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
if not Path('baselines/Retinexformer').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/caiyuanhao1998/Retinexformer.git','baselines/Retinexformer'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','einops','gdown','lmdb','yacs','addict','future','natsort','yapf','thop','timm','lpips','scikit-image'],check=True)

import torch
assert torch.cuda.is_available() and torch.cuda.device_count()>=2, 'Enable GPU T4 x2'
OPT='baselines/Retinexformer/Options/RetinexFormer_LOL_v1.yml'  # identical network_g to syn yml; matches prior runs
RUNS=Path('runs/gtmean_gate')
print('GPUs:',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


## Restore state (if resuming)


In [ ]:
import shutil
zips=glob.glob('/kaggle/input/**/gtmean_gate_state*.zip',recursive=True)
if zips:
    z=max(zips,key=os.path.getsize)
    with zipfile.ZipFile(z) as zf: zf.extractall('/kaggle/working/btp')
    print('Restored',z)
else:
    # Kaggle may unpack an uploaded zip into a dataset - use the extracted runs/ directly.
    found=glob.glob('/kaggle/input/**/runs/gtmean_gate/*/metrics.json',recursive=True)
    if found:
        source=Path(found[0]).parents[2]   # .../runs
        shutil.copytree(source,Path('runs'),dirs_exist_ok=True)
        print('Restored extracted runs from',source)
    else:
        print('Fresh run')

def save_state():
    p='/kaggle/working/gtmean_gate_state.zip'
    with zipfile.ZipFile(p,'w',zipfile.ZIP_DEFLATED) as zf:
        for f in RUNS.rglob('*'):
            if f.is_file(): zf.write(f,f.as_posix())
        if Path('gate.json').exists(): zf.write('gate.json','gate.json')
    print('Saved',p)


## Data + official LOL-v2-syn weights


In [ ]:
def gdl(fid,out,tries=4):
    out=Path(out); out.parent.mkdir(parents=True,exist_ok=True)
    for _ in range(tries):
        if subprocess.run(['gdown','-q',fid,'-O',str(out)]).returncode==0 and out.exists(): return
    raise RuntimeError('download failed '+str(out))

DATA=Path('data/LOLv2/Synthetic')
if not (DATA/'Train/Low').is_dir():
    attached=glob.glob('/kaggle/input/**/LOLv2/Synthetic/Train/Low',recursive=True)
    if attached:
        src=Path(attached[0]).parents[2]; Path('data').mkdir(exist_ok=True)
        if not Path('data/LOLv2').exists(): Path('data/LOLv2').symlink_to(src,target_is_directory=True)
    else:
        gdl('1Ou9EljYZW8o5dbDCf9R34FS8Pd8kEp2U','/kaggle/tmp/lolv2.zip')
        subprocess.run(['unzip','-q','-o','/kaggle/tmp/lolv2.zip','-d','data'],check=True)
assert len(list((DATA/'Train/Low').glob('*')))==900

WDIR=Path('baselines/Retinexformer/pretrained_weights')
W=WDIR/'LOL_v2_synthetic.pth'
if not W.exists():
    c=list(WDIR.glob('**/LOL_v2_synthetic.pth'))
    if not c:
        subprocess.run(['gdown','--folder','1ynK5hfQachzc8y96ZumhkPPDXzHJwaQV','-O','/kaggle/tmp/rf_weights'],check=True)
        c=list(Path('/kaggle/tmp/rf_weights').glob('**/LOL_v2_synthetic.pth'))
    W.parent.mkdir(parents=True,exist_ok=True)
    import shutil; shutil.copyfile(str(c[0]),str(W))
W=str(W); print('weights:',W)


## Six matched 10k runs — two per GPU


In [ ]:
tasks=[]
for loss in ('l1','gtmean'):
    for seed in (42,43,44):
        tasks.append({'loss':loss,'seed':seed,'iters':10000,
                      'out':str(RUNS/f'{loss}_seed{seed}')})

N_GPU = torch.cuda.device_count()
def run_pair(tasks,timeout=5000):
    # Only N_GPU processes at a time; assigning GPU ids beyond the available devices
    # gives "No CUDA GPUs are available" on Kaggle's T4 x2.
    for offset in range(0,len(tasks),N_GPU):
        batch=[t for t in tasks[offset:offset+N_GPU] if not (Path(t['out'])/'metrics.json').exists()]
        if not batch:
            continue
        launched=[]
        for gpu,t in enumerate(batch):
            out=Path(t['out']); out.mkdir(parents=True,exist_ok=True)
            cmd=['timeout',str(timeout),sys.executable,'scripts/finetune.py',
                 '--loss',t['loss'],'--iters',str(t['iters']),'--lr','2e-5',
                 '--batch','8','--crop','128','--seed',str(t['seed']),
                 '--pretrained',W,'--opt',OPT,'--repo','baselines/Retinexformer',
                 '--data',str(DATA/'Train'),'--device','cuda',
                 '--val-fraction','0.1','--val-seed','2026','--val-every','1000',
                 '--checkpoint-every','5000','--log-every','500','--out',str(out)]
            if (out/'latest_state.pth').exists(): cmd += ['--resume']
            env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=str(gpu)
            log=(out/'run.log').open('a')
            launched.append((t,subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,env=env),log))
        for t,proc,log in launched:
            rc=proc.wait(); log.close(); print(t['out'],'exit',rc)
            if rc not in (0,124): raise RuntimeError((Path(t['out'])/'run.log').read_text()[-4000:])

try:
    run_pair(tasks)
finally:
    save_state()   # always save before reading metrics


## Gate on validation PSNR only


In [ ]:
scores={}
for t in tasks:
    m=Path(t['out'])/'metrics.json'
    if m.exists():
        scores[(t['loss'],t['seed'])]=json.loads(m.read_text())['best_val_psnr']
for k,v in sorted(scores.items()): print(k,round(v,4))

l1=[v for (l,s),v in scores.items() if l=='l1']
gm=[v for (l,s),v in scores.items() if l=='gtmean']
if len(l1)==3 and len(gm)==3:
    mean_l1=sum(l1)/3; mean_gm=sum(gm)/3
    gain=mean_gm-mean_l1
    passed=gain>0.1
    print(f'mean L1 {mean_l1:.4f} | mean GTmean {mean_gm:.4f} | gain {gain:+.4f} dB')
    print('GATE PASSED' if passed else 'GATE FAILED')
    Path('gate.json').write_text(json.dumps({
        'scores':{f'{l}_seed{s}':v for (l,s),v in scores.items()},
        'mean_l1':mean_l1,'mean_gtmean':mean_gm,'gain_db':gain,'passed':passed},indent=2))
    save_state()
    if not passed:
        print('STOP: keep the Day 1 no-training GT-Mean + self-ensemble result as final.')
else:
    print('Incomplete runs: attach the saved state zip and run again.')


## Done

Download `gtmean_gate_state.zip` and `gate.json`. If the gate passed, I will prepare the Day 3/4 longer
fine-tune notebook. If it failed, the paper is final with the Day 1 result.
